----
# **<span style="color:DarkSlateBlue">PROYECTO BOOKING_REVIEWS</span>**
# **<span style="color:DarkSlateBlue">Limpieza de datos</span>**
----

---
---
## <span style="color:gray">**1. Importación de librerías**</span> 📂

In [40]:
# Tratamiento de datos
import numpy as np
import pandas as pd 
from IPython.display import display
import pycountry # Mapea códigos ISO a nombres completos
import ast # Convierte strings literales a objetos Python de forma segura

# Visualización de datos
import seaborn as sns
import matplotlib.pyplot as plt

# Configuración de ruta
import sys
sys.path.append('../')

# Importación de funciones personalizadas
from src.soporte2_limpieza import *


---
---
## <span style="color:gray">**2. Carga de datos**</span> 📥

In [41]:
booking_raw = pd.read_csv("../data/raw/hotel_booking.csv")

reviews_raw = pd.read_csv("../data/raw/hotel_reviews.csv")

In [42]:
# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)

# Configuración del ancho de las columnas
pd.set_option('display.max_colwidth', 80)

---
---

----
# <span style="color:DarkSlateBlue">**Desarrollo del proyecto - 2**</span>
----


---
---
## <span style="color:gray">**Limpieza de datos**</span> 🧹

Antes de continuar con el análisis exploratorio de datos (EDA), creamos copias de los DataFrames originales.
Esto nos permite manipular, limpiar o transformar los datos sin afectar los originales, asegurando que siempre podamos recuperar la información original si es necesario.

In [43]:
booking_limpio = booking_raw.copy()
reviews_limpio = reviews_raw.copy()

---
## `booking_limpio`

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>


La única columna que habría que estandarizar es `phone-number`, ya que utiliza como separador "-", en lugar de "_" como el resto de nombres de columnas. Sin embargo, al ser irrelevante para nuestro análisis, la eliminaremos en el próximo paso.

### <span style="color:darkgray">**2. Eliminación de columnas irrelevantes**</span>

Eliminamos las columnas que hemos mencionado en el análisis preliminar.

In [44]:
# Llamamos a la función para eliminar columnas de un DataFrame guardada en el archivo de soporte
booking_limpio = eliminar_columns(
    booking_limpio,
    ["arrival_date_week_number", "arrival_date_day_of_month", 
     "agent", "company", "days_in_waiting_list", 
     "required_car_parking_spaces", "total_of_special_requests",
     "name", "email", "phone-number", "credit_card"]
)

### <span style="color:darkgray">**3. Limpieza de strings**</span>

- **Cambiamos la abreviatura de los países por el nombre completo del país.**

In [45]:
# Aplicamos la función guardada en el documento de soporte para convertir códigos ISO alpha-3 a nombres de países
booking_limpio['country'] = booking_limpio['country'].apply(code_to_name)

- **Comprobamos los datos únicos de las variables categóricas y hacemos la limpieza de algunos de los datos.**

In [46]:
col_cat = booking_limpio.select_dtypes(include=['category', 'str']).columns
datos_unicos(booking_limpio, col_cat)



Los datos únicos de la varible hotel son:

 <StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible arrival_date_month son:

 <StringArray>
[     'July',    'August', 'September',   'October',  'November',  'December',
   'January',  'February',     'March',     'April',       'May',      'June']
Length: 12, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible meal son:

 <StringArray>
['BB', 'FB', 'HB', 'SC', 'Undefined']
Length: 5, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible country son:

 <StringArray>
[                        'Portugal',                   'United Kingdom',
                    'United States',                            'Spain',
                          'Ireland',                           'France',
                             

- Todas las variables presentan mayúsculas y minúsculas, lo estandarizamos todo a minúsculas.
- La variable `distribution_channel` puede eliminarse porque `market_segment` ya cubre la información de origen de la reserva.
- Las variables `hotel`, `country`, `deposit_type` y `market_segment` tienen algunos datos con espacio entre medias, lo cambiamos por "_".
- `market_segment` tiene el caracter "/", el cual sustituimos por "_".
- `customer_type` y `reservation_status` tienen como separador de palabras "-", lo cambiamos por "_".


- Se identifica que las variables `reservation_status` e `is_canceled` podrían contener información redundante. Para verificar esta hipótesis, se analiza la relación entre ambas mediante una tabla de contingencia:

In [47]:
pd.crosstab(booking_limpio['reservation_status'], booking_limpio['is_canceled'])

is_canceled,0,1
reservation_status,,
Canceled,0,43017
Check-Out,75166,0
No-Show,0,1207


Se verifica mediante la tabla de contingencia que existe una correspondencia perfecta entre ambas variables: los valores de `reservation_status` determinan completamente `is_canceled`. Esto evidencia una redundancia total de información.

Mantener ambas variables introduce multicolinealidad sin aportar valor adicional al modelo. Por ello, se decide eliminar `is_canceled` y conservar `reservation_status`, ya que aporta mayor riqueza semántica al incluir más estados además de la cancelación.

In [48]:
# Llamamos a la función que transforma datos a minúscula guardada en el archivo de sorporte
minus(booking_limpio, col_cat)

# Llamamos a la función para eliminar columnas
booking_limpio = eliminar_columns(booking_limpio, ["distribution_channel", "is_canceled"])

# Llamamos a la función reemplazar guardada en el archivo de soporte 
reemplazar(booking_limpio, ["hotel", "country", "deposit_type", "market_segment"], " ", "_")
reemplazar(booking_limpio, ["market_segment"], "/", "_")
reemplazar(booking_limpio, ["customer_type", "reservation_status"], "-", "_")

### <span style="color:darkgray">**4. Corrección de tipos de datos**</span>

- **Cambiamos la variable `is_repeated_guest` a variable categórica:** 

   - 1 = 'yes'
   - 0 = 'no'

In [49]:

booking_limpio["is_repeated_guest"] = booking_limpio["is_repeated_guest"].replace({1:'yes', 0:'no'})

- **Variable `children`**

Convertimos esta variable a números enteros:

In [50]:
booking_limpio['children'] = booking_limpio['children'].astype('Int64')

- **Variable `reservation_status_date`:** Cambiamos el tipo de dato a Datetime

In [51]:
booking_limpio["reservation_status_date"] = (
    pd.to_datetime(booking_limpio["reservation_status_date"], 
                   format="%Y-%M-%d", 
                   errors="coerce")
                   )

### <span style="color:darkgray">**5. Tratamiento de valores nulos**</span>

Durante la exploración inicial del Dataset, identificamos columnas con valores nulos; a continuación, evaluaremos cómo tratarlos individualmente.

Recordamos las variables que contienen nulos:

In [52]:
nulos(booking_limpio)

Las columnas con nulos son: ['children', 'country']

El porcentaje de nulos de cada columna es:
children    0.003
country     1.483
dtype: float64


- Variable `children`

Se imputa la moda debido a que se trata de una variable discreta con muy pocos valores faltantes, lo que permite mantener la distribución original sin introducir cambios significativos.

In [53]:
booking_limpio["children"] = booking_limpio["children"].fillna(booking_limpio["children"].mode()[0])

- Variable `country`

Se reemplazan los valores nulos por la categoría "unknown". Esta decisión evita distorsionar la distribución geográfica del dataset y permite conservar la información de “dato desconocido” de forma explícita.

In [54]:
booking_limpio["country"] = booking_limpio["country"].fillna('unknown') 

### <span style="color:darkgray">**6. Feature Engineering:**</span>

En esta etapa se crean variables derivadas con el objetivo de mejorar la calidad del análisis y facilitar la interpretación del comportamiento de las reservas hoteleras.

- `arrival_year_month` → Mes y año de llegada al hotel.

    Esta variable combina el año y el mes de llegada en una sola característica temporal. Su objetivo es capturar patrones estacionales de forma más directa que usando variables separadas. Esto permite analizar tendencias temporales en las reservas y detectar estacionalidad en la ocupación del hotel.

    Además, esta variable se crea con un propósito clave dentro del proyecto: facilitar la integración entre el dataset de reservas y el de reseñas. Posteriormente, se utilizará junto con la variable `country` como clave para realizar el *merge* entre ambos datasets, asegurando que la información quede alineada tanto temporal como geográficamente.

In [55]:
booking_limpio["arrival_year_month"] = pd.to_datetime(
    booking_limpio["arrival_date_month"] + "-" +
    booking_limpio["arrival_date_year"].astype(str), 
    format="%B-%Y"
)

Eliminamos `arrival_date_month` y `arrival_date_year`

In [56]:
booking_limpio = eliminar_columns(booking_limpio, ["arrival_date_month", "arrival_date_year"])

- `total_nights` → Número de noches totales. 

    Esta variable resume la duración total de la estancia, sumando noches de fin de semana y entre semana. Es una métrica clave para entender el comportamiento del cliente (estancias cortas vs largas) y analizar patrones de consumo hotelero.

In [57]:
booking_limpio['total_nights'] = (
    booking_limpio['stays_in_weekend_nights'] + 
    booking_limpio['stays_in_week_nights']
)

Aunque se creó la variable `total_nights` para simplificar el análisis de la duración de la estancia, se decidió mantener las variables originales (`stays_in_weekend_nights` y `stays_in_week_nights`) ya que aportan información adicional sobre el tipo de estancia (ocio vs negocio), lo que puede resultar relevante en análisis posteriores.

- `total_guests` → Total de personas por reserva.
    Se calcula como la suma de adultos, niños y bebés. Esta variable permite medir el tamaño del grupo asociado a cada reserva, lo cual es útil para segmentación de clientes y análisis de demanda (parejas, familias, grupos).

In [58]:
booking_limpio['total_guests'] = (
    booking_limpio['adults'] + 
    booking_limpio['children'] + 
    booking_limpio['babies']
)

Aunque se creó la variable `total_guests` para representar el tamaño total del grupo, se decidió mantener las variables originales (`adults`, `children` y `babies`), ya que permiten identificar el tipo de cliente (familias, parejas, viajeros individuales) y aportan información relevante para la segmentación y el análisis del comportamiento de la demanda.

- `european_countries` → Filtrado de países europeos

    Dado que el dataset de reseñas está centrado exclusivamente en hoteles europeos, se filtran únicamente las reservas correspondientes a estos países. Esto asegura coherencia geográfica en el análisis y evita ruido introducido por observaciones fuera del dominio de estudio. 

    Este filtrado se realizará en una etapa posterior, una vez dispongamos de la variable `country` en el dataset de reseñas. De este modo, se pueden identificar exactamente los países comunes entre ambos datasets y trabajar únicamente con ese subconjunto, garantizando una correcta integración de la información.

---
## `reviews_limpio`

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>

In [59]:
estandar_columns(reviews_limpio)

### <span style="color:darkgray">**2. Eliminación de columnas irrelevantes**</span>

In [60]:
# Llamamos a la función para eliminar columnas de un DataFrame guardada en el archivo de soporte
reviews_limpio = eliminar_columns(
    reviews_limpio, 
    ["review_total_negative_word_counts", 
     "review_total_positive_word_counts",
     "days_since_review", 
     "lat", "lng"]
     )

### <span style="color:darkgray">**3. Limpieza de strings**</span>

- **Cambiamos los datos de las variables categóricas a minúscula.**

In [61]:
# Llamamos a la función que transforma datos a minúscula guardada en el archivo de sorporte
col_cat = reviews_limpio.select_dtypes(include=['category', 'str']).columns
minus(reviews_limpio, col_cat)

- **Normalización de `reviewer_nationality`.**

In [62]:
reemplazar(reviews_limpio, ["reviewer_nationality"], " ", "_")

### <span style="color:darkgray">**4. Corrección de tipos de datos**</span>

In [63]:
reviews_limpio['review_date'] = pd.to_datetime(reviews_limpio['review_date'])

### <span style="color:darkgray">**5. Tratamiento de valores nulos**</span>


Durante el análisis preliminar del dataset, se identificó que las únicas variables que contenían valores nulos eran `lat` y `lng`, correspondientes a la latitud y longitud de los hoteles. Dado que se optó por eliminar estas variables, el dataset queda completamente libre de valores nulos, lo que simplifica el procesamiento y evita la necesidad de aplicar técnicas adicionales de imputación o tratamiento de datos faltantes.

### <span style="color:darkgray">**6. Eliminación de duplicados**</span>

In [64]:
reviews_limpio = reviews_limpio.drop_duplicates()

### <span style="color:darkgray">**7. Feature Engineering**</span>

- **Variables `country` y `city`.**

    Esto permite organizar la información geográfica de forma más clara y resulta fundamental para el análisis por localización. En particular, la variable `country` será clave en la posterior integración con el dataset de reservas.

In [65]:
reviews_limpio['country'] = reviews_limpio['hotel_address'].apply(lambda x: x.split()[-1])
reviews_limpio['city'] = reviews_limpio['hotel_address'].apply(lambda x: x.split()[-2])

- **Transformación de la variable `review_date` para obtener `review_year_month`.**

    Conservamos únicamente el mes y el año. Esto reduce la granularidad temporal y permite analizar tendencias de forma más clara. Además, esta variable será fundamental para alinear temporalmente ambos datasets durante el proceso de integración, junto con `arrival_year_month`.

In [66]:
reviews_limpio['review_year_month'] = reviews_limpio['review_date'].dt.strftime('%Y-%m')

- **Variables `positive_reviews` y `negative_reviews`.**

    Estas columnas, al contener texto libre, se simplifican mediante la extracción de palabras clave. Esta transformación permite reducir la complejidad del texto y convertirlo en información más estructurada, facilitando la identificación de patrones en las opiniones de los clientes.

In [67]:
# Aplicar la función guardada en el documneto de soporte para clasificar reviews a las columnas de reviews
reviews_limpio['positive_tag'] = reviews_limpio['positive_review'].apply(clasificar_review)
reviews_limpio['negative_tag'] = reviews_limpio['negative_review'].apply(clasificar_review)

- **Variable `tags`.**

    A partir de esta columna, que contiene múltiples características en formato no estructurado, se extraen las siguientes variables: 
    
    - `trip_type` → motivo del viaje (ocio, negocio, etc.)
    - `group_type` → tipo de cliente (pareja, familia, grupo…)
    - `room_type` → tipo de habitación reservada
    - `nights` → duración de la estancia
    - `device` → desde qué dispositivo se realizó la reserva

    Esta transformación permite descomponer información compleja en variables individuales, facilitando la segmentación de clientes y el análisis del comportamiento según el tipo de viaje, grupo o condiciones de la reserva.


In [68]:
# Cambiamos los string a lista de strings
reviews_limpio['tags'] = reviews_limpio['tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [69]:
# Aplicamos la función procesar_tags guardada en el documento de soporte a la variable `tags`
reviews_limpio = reviews_limpio.join(reviews_limpio['tags'].apply(procesar_tags))

Eliminamos las columnas a partir de las cuales hemos creado las nuevas variables

In [70]:
reviews_limpio = eliminar_columns(
    reviews_limpio, ["hotel_address", "review_date", "positive_review", "negative_review", "tags"])

---


Ahora que ya disponemos de la variable `country` en el dataset de reseñas, podemos filtrar el dataset de reservas para quedarnos únicamente con los países coincidentes entre ambas fuentes de datos.

Sin embargo, antes de aplicar este filtrado, se crea una copia del dataset de reservas y posteriormente se guarda en un archivo CSV con el nombre 'booking_global' con el objetivo de conservar la información completa de todos los países y evitar la pérdida de datos para futuros análisis.

In [71]:
# Creamos la copia
booking_eu = booking_limpio.copy()

In [72]:
# Guardamos el dataset con todos los países
booking_limpio.to_csv("../data/output/booking_global.csv", index=False)

In [73]:
# Lista de países únicos del dataset de reseñas
eu_countries = reviews_limpio['country'].unique()

# Filtrado del dataset de reservas
booking_eu = booking_eu[
    booking_eu['country'].isin(eu_countries)
]

Este proceso asegura la coherencia geográfica entre ambos datasets, lo que permite su posterior integración y análisis conjunto.

---
---
## <span style="color:gray">**Validación final de los datasets**</span> ✅

- **Comprobamos que la limpieza de ambos dataset se ha realizado correctamente**


### **`booking_eu`** 

In [74]:
# Llamamos a la función analisis_rapido guardada en el archivo de soporte
analisis_rapido(booking_eu)



Las 3 primeras filas del Dataframe son:



,hotel,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,customer_type,adr,reservation_status,reservation_status_date,arrival_year_month,total_nights,total_guests
13,resort_hotel,18,0,4,2,1,0,hb,spain,online_ta,no,0,0,g,g,1,no_deposit,transient,154.77,check_out,2015-01-05 00:07:00,2015-07-01,4,3
18,resort_hotel,0,0,1,2,0,0,bb,france,corporate,no,0,0,a,g,0,no_deposit,transient,107.42,check_out,2015-01-02 00:07:00,2015-07-01,1,2
36,resort_hotel,15,1,3,2,0,0,bb,spain,online_ta,no,0,0,a,c,0,no_deposit,transient,98.00,check_out,2015-01-06 00:07:00,2015-07-01,4,2



-----------------------------------------------------------

La información básica del Dataframe es la siguiente:

<class 'pandas.DataFrame'>
Index: 26116 entries, 13 to 119386
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           26116 non-null  str           
 1   lead_time                       26116 non-null  int64         
 2   stays_in_weekend_nights         26116 non-null  int64         
 3   stays_in_week_nights            26116 non-null  int64         
 4   adults                          26116 non-null  int64         
 5   children                        26116 non-null  Int64         
 6   babies                          26116 non-null  int64         
 7   meal                            26116 non-null  str           
 8   country                         26116 non-null  str           
 9   market_segment                  2611

hotel                             0
lead_time                         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
customer_type                     0
adr                               0
reservation_status                0
reservation_status_date           0
arrival_year_month                0
total_nights                      0
total_guests                      0
dtype: int64

En las variables `reservation_status_date` y `arrival_year_month` se observa que la información incluye un nivel de detalle diario (y en el caso de `reservation_status_date`, también horario). Sin embargo, para el análisis no es necesario trabajar a este nivel de granularidad.

Por ello, se transforma ambas variables para conservar únicamente el mes y el año. Esta simplificación permite un análisis temporal más consistente y facilita la integración y comparación entre distintos datasets, reduciendo el ruido asociado a la variación diaria.

In [75]:
booking_eu["reservation_status_date"] = booking_eu["reservation_status_date"].dt.strftime("%Y-%m")

In [76]:
booking_eu["arrival_year_month"] = booking_eu["arrival_year_month"].dt.strftime("%Y-%m")

### **`reviews_limpio`** 

In [ ]:
# Llamamos a la función analisis_rapido guardada en el archivo de soporte
analisis_rapido(reviews_limpio)

trip_type y group_type utilizan espacio como separado, lo cambiamos por "_"

In [ ]:
reemplazar(reviews_limpio, ["trip_type", "group_type"], " ", "_")

nights nos quedamos solo con el número y cambiamos el tipo de dato a int

In [ ]:
reemplazar(reviews_limpio, ["nights"], "stayed", "")
reemplazar(reviews_limpio, ["nights"], ("night"), "")
reemplazar(reviews_limpio, ["nights"], "s", "")


In [ ]:
reviews_limpio["nights"] = reviews_limpio["nights"].str.strip().astype(int)

In [ ]:
minus(reviews_limpio, ['review_month'])

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
- `room_type` lo eliminamos porque hay muchos datos únicos y no es información relevante para nuestro análisis

- `device` el único dato que tenemos es *mobile*, así que como no disponemos información del resto de devices, la eliminamos

In [ ]:
reviews_limpio = eliminar_columns(reviews_limpio, ["room_type", "device"])

- trip_type lo rellenamos con *unknown*

In [ ]:
reviews_limpio["trip_type"] = reviews_limpio["trip_type"].fillna('unknown')

- nights lo rellenamos con la moda

In [ ]:
reviews_limpio["nights"] = reviews_limpio["nights"].fillna(reviews_limpio["nights"].mode()[0])

Hemos verificado que la limpieza de ambos dataframes se ha realizado correctamente: 

- Los nombres de las columnas son claros y descriptivos.
- Los datos son consistentes entre sí.
- No se observan valores nulos.
- Las variables presentan formatos adecuados para análisis y modelado posteriores .

Además, las transformaciones aplicadas (como estandarización de texto, conversión de tipos numéricos y codificación de variables categóricas) aseguran que ambos dataframes puedan combinarse y utilizarse de manera confiable.

- **Guardamos los dataset limpios**

In [ ]:
booking_limpio.to_csv("../data/output/booking_limpio.csv", index=False)
reviews_limpio.to_csv("../data/output/reviews_limpio.csv", index=False)

---

In [ ]:
booking_limpio.columns

In [ ]:
reviews_limpio.columns